In [ ]:
from pathlib import Path

import folium
import geopandas as gpd
import numpy as np
import rasterio
from matplotlib import pyplot as plt
from rasterio.mask import mask as rmask
from skimage import exposure

from estuary.util.img import contrast_stretch

In [ ]:
BASE = Path("/Users/kyledorman/data/estuary_poc/")

for p in BASE.iterdir():
    if p.is_dir():
        print(p.stem)

In [ ]:
def s2_broad_band(bands):
    blue = np.log10(1 + bands[10:12].mean(axis=0))
    green = np.log10(1 + bands[0:3].mean(axis=0))
    red = np.log10(1 + bands[3:9].mean(axis=0))

    img = np.dstack([red, green, blue])
    return contrast_stretch(img.transpose((2, 0, 1))).transpose((1, 2, 0))


def s2_tri_stimulus(bands):
    red_recipe = np.log10(
        1.0
        + 0.01 * bands[0]
        + 0.09 * bands[1]
        + 0.35 * bands[2]
        + 0.04 * bands[3]
        + 0.01 * bands[4]
        + 0.59 * bands[5]
        + 0.85 * bands[6]
        + 0.12 * bands[7]
        + 0.07 * bands[9]
        + 0.04 * bands[10]
    )
    green_recipe = np.log10(
        1.0
        + 0.26 * bands[2]
        + 0.21 * bands[3]
        + 0.50 * bands[4]
        + 1.00 * bands[5]
        + 0.38 * bands[6]
        + 0.04 * bands[7]
        + 0.03 * bands[9]
        + 0.02 * bands[10]
    )
    blue_recipe = np.log10(
        1.0
        + 0.07 * bands[0]
        + 0.28 * bands[1]
        + 1.77 * bands[2]
        + 0.47 * bands[3]
        + 0.16 * bands[4]
    )

    rgb_tri = np.dstack((red_recipe, green_recipe, blue_recipe))
    mn = rgb_tri.min()
    rgb_tri -= mn
    rgb_tri /= rgb_tri.max()
    return contrast_stretch(rgb_tri.transpose((2, 0, 1))).transpose((1, 2, 0))


S2_BAND_NAMES = [
    "B01",
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B09",
    "B11",
    "B12",
]


def s2_img(s2_dir):
    bands = []
    for name in S2_BAND_NAMES:
        file = list(s2_dir.glob(f"*_{name}_*"))[0]
        with rasterio.open(file) as src:
            bands.append(src.read(1))

    return s2_broad_band(np.array(bands))

In [ ]:
def plot_grid(region):
    gdf = gpd.read_file(BASE / region / "grid.geojson").to_crs("wgs84")
    gdf["filename"] = region

    # Compute the bounding box of all polygons
    minx, miny, maxx, maxy = gdf.total_bounds

    # Calculate the center of the bounding box
    center_lat = (miny + maxy) / 2
    center_lon = (minx + maxx) / 2

    # Step 5: Create a Folium Map and Overlay
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

    for _, row in gdf.iterrows():
        folium.GeoJson(
            row.geometry,
            name=row.filename,
            # tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Region:"]),
            popup=folium.Popup(row.filename, parse_html=True),
            style_function=lambda x: {
                "fillColor": "red",
                "color": "black",
                "weight": 1,
                "fillOpacity": 0.5,
            },
        ).add_to(m)

    # Display the map (if running in a Jupyter Notebook)
    return m


def plot_images(region, width, height):
    dove_images = sorted(list((BASE / region / "dove" / "images").iterdir()))
    skysat_images = sorted(list((BASE / region / "skysat" / "images").iterdir()))
    sentinel_image_dirs = sorted(list((BASE / region / "sentinel").iterdir()))

    rows = len(skysat_images)
    cols = 3

    fig, axs = plt.subplots(rows, cols, figsize=(width * cols, height * rows))
    for ax in axs.flatten():
        ax.axis("off")

    gdf = gpd.read_file(BASE / region / "grid.geojson")

    for i, (dove_image_path, skysat_image_path) in enumerate(
        zip(dove_images, skysat_images, strict=False)
    ):
        for c, (pth, name) in enumerate([(skysat_image_path, "SkySat"), (dove_image_path, "Dove")]):
            axs[i, c].set_title(
                pth.name[:4] + "-" + pth.name[4:6] + "-" + pth.name[6:8] + f" {name}"
            )

            with rasterio.open(pth) as src:
                # Reproject polygon to match raster CRS if needed
                if gdf.crs != src.crs:
                    gdf = gdf.to_crs(src.crs)

                geo = gdf.geometry

                # Apply mask with crop=True and filled=True
                data, out_transform = rmask(src, geo, crop=True, filled=True, nodata=0)
                mask = np.all(data == 0, axis=0)

                img = false_color_log(data, mask)
                img = np.uint8(exposure.equalize_adapthist(img) * 255)

                axs[i, c].imshow(img)

    for i, pth in enumerate(sentinel_image_dirs):
        axs[i, 2].set_title(pth.name[:10] + " Sentinel")
        axs[i, 2].imshow(s2_img(pth))

    fig.tight_layout()


# plot_images("malibu", width=5, height=7)

# ventura

In [ ]:
plot_grid("ventura")

In [ ]:
plot_images("ventura", width=5, height=7)

# topanga

In [ ]:
plot_grid("topanga")

In [ ]:
plot_images("topanga", width=5, height=6)

# malibu

In [ ]:
plot_grid("malibu")

In [ ]:
plot_images("malibu", width=7, height=5)

# goleta

In [ ]:
plot_grid("goleta")

In [ ]:
plot_images("goleta", width=6, height=4)

# arroyo_sequit

In [ ]:
plot_grid("arroyo_sequit")

In [ ]:
plot_images("arroyo_sequit", width=5, height=5)